In [3]:
import itertools
import re
import os
import pandas as pd
import gc
import torch
from transformers import pipeline

2026-07-27 16:02:27.982677: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-27 16:02:28.128121: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1785160948.232268 2462099 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1785160948.259239 2462099 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1785160948.445324 2462099 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

/home/muti/miniconda3/envs/testenv/lib/python3.11/site-packages/tensorflow/python/keras/engine/training_arrays_v1.py:37: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.4.6)
  from scipy.sparse import issparse  # pylint: disable=g-import-not-at-top


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [4]:
df = pd.read_csv('combined_prototypes.csv')

In [5]:
LANGUAGE = "German"

instance_col = f"instance_{LANGUAGE}"
norm_rating_col = f"norm_rating_{LANGUAGE}"
# keep only rows with a non-empty instance for the selected language
df = df[df[instance_col].notna() & (df[instance_col].astype(str).str.strip() != "")].reset_index(drop=True)

df.head(5)

,category,concept_en,instance_English,instance_German,instance_Spanish,norm_rating_English,norm_rating_German,norm_rating_Spanish,n_languages,in_multiple_languages
0,animal,ANT,ant,Ameise,Hormiga,0.308662,0.630556,0.521127,3,True
1,animal,BEE,bee,Biene,Abeja,0.309422,0.711111,0.521127,3,True
2,animal,BUTTERFLY,butterfly,Schmetterling,Mariposa,0.298172,0.746667,0.507042,3,True
3,animal,CROW,crow,Krähe,Cuervo,0.499310,0.820556,0.394366,3,True
4,animal,DUCK,duck,Ente,Pato,0.772608,0.936667,0.830986,3,True


In [6]:
#PUT YOUR MODEL HERE, GEMMA OR LLAMA
import torch, gc
torch.cuda.empty_cache()
gc.collect()
from transformers import pipeline

MODEL_NAME = "/data1/shared_models/models--google--gemma-3-12b-it/snapshots/96b6f1eccf38110c56df3a15bffe176da04bfd80//"
#MODEL_NAME = "/data1/shared_models/models--meta-llama--Llama-3.1-8B-Instruct/snapshots/0e9e39f249a16976918f6564b8830bc894c89659/"

pipe = pipeline(
    "text-generation",
    model=MODEL_NAME,
    tokenizer=MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

The module name  (originally ) is not a valid Python identifier. Please rename the original module to avoid import issues.
Device set to use cuda:0


In [7]:
def pairwise_prompt(a, b, category):
    return f"""
Sie nehmen an einem psychologischen Experiment teil.

Welches ist ein typischeres Beispiel für die Kategorie „{category}“: „{a}“ oder „{b}“?
Antworten Sie NUR mit einem Buchstaben: A oder B.
A = „{a}“
B = „{b}“
"""

In [10]:
def get_pairwise_choice(a, b, category):
    prompt = pairwise_prompt(a, b, category)
    messages = [{"role": "user", "content": prompt}]
    output = pipe(messages, max_new_tokens=10, do_sample=False)
    content = output[0]["generated_text"][-1]["content"]
    match = re.search(r"\b([AB])\b", content.strip())
    if match:
        letter = match.group(1)
        return a if letter == "A" else b
    else:
        print(f"Could not parse response for '{a}' vs '{b}' ({category}): {content!r}")
        return None

In [11]:
output_path = f"pairwise_typicality_GEMMA_{LANGUAGE}_de.csv"

# resume support: load existing results if the file already exists
if os.path.exists(output_path):
    results_df = pd.read_csv(output_path)
    done_pairs = set(zip(results_df["item_a"], results_df["item_b"], results_df["category"]))
else:
    results_df = pd.DataFrame(columns=["category", "item_a", "item_b", "winner"])
    done_pairs = set()

rows = []
for category, group in df.groupby("category"):
    instances = group[instance_col].tolist()
    for a, b in itertools.combinations(instances, 2):
        if (a, b, category) in done_pairs:
            continue
        winner = get_pairwise_choice(a, b, category)
        row = {"category": category, "item_a": a, "item_b": b, "winner": winner}
        rows.append(row)

        # append + save incrementally so progress survives interruptions
        pd.DataFrame([row]).to_csv(
            output_path, mode="a", header=not os.path.exists(output_path), index=False
        )

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [12]:
LANGUAGE = "Spanish"

instance_col = f"instance_{LANGUAGE}"
norm_rating_col = f"norm_rating_{LANGUAGE}"

In [13]:
def pairwise_prompt(a, b, category):
    return f"""
Está participando en un experimento de psicología.

¿Cuál es un ejemplo más típico de la categoría «{category}»: «{a}» o «{b}»?
Responda SOLO con una letra: A o B.
A = «{a}»
B = «{b}»
"""

In [14]:
output_path = f"pairwise_typicality_GEMMA_{LANGUAGE}_es.csv"

# resume support: load existing results if the file already exists
if os.path.exists(output_path):
    results_df = pd.read_csv(output_path)
    done_pairs = set(zip(results_df["item_a"], results_df["item_b"], results_df["category"]))
else:
    results_df = pd.DataFrame(columns=["category", "item_a", "item_b", "winner"])
    done_pairs = set()

rows = []
for category, group in df.groupby("category"):
    instances = group[instance_col].tolist()
    for a, b in itertools.combinations(instances, 2):
        if (a, b, category) in done_pairs:
            continue
        winner = get_pairwise_choice(a, b, category)
        row = {"category": category, "item_a": a, "item_b": b, "winner": winner}
        rows.append(row)

        # append + save incrementally so progress survives interruptions
        pd.DataFrame([row]).to_csv(
            output_path, mode="a", header=not os.path.exists(output_path), index=False
        )

In [15]:
LANGUAGE = "English"

instance_col = f"instance_{LANGUAGE}"
norm_rating_col = f"norm_rating_{LANGUAGE}"

In [16]:
def pairwise_prompt(a, b, category):
    return f"""
You are participating in a psychology experiment. 
Which is a more typical example of the category "{category}": "{a}" or "{b}"? 
Respond with ONLY one letter: A or B. A = "{a}" B = "{b}"
"""

In [17]:
output_path = f"pairwise_typicality_GEMMA_{LANGUAGE}_en.csv"

# resume support: load existing results if the file already exists
if os.path.exists(output_path):
    results_df = pd.read_csv(output_path)
    done_pairs = set(zip(results_df["item_a"], results_df["item_b"], results_df["category"]))
else:
    results_df = pd.DataFrame(columns=["category", "item_a", "item_b", "winner"])
    done_pairs = set()

rows = []
for category, group in df.groupby("category"):
    instances = group[instance_col].tolist()
    for a, b in itertools.combinations(instances, 2):
        if (a, b, category) in done_pairs:
            continue
        winner = get_pairwise_choice(a, b, category)
        row = {"category": category, "item_a": a, "item_b": b, "winner": winner}
        rows.append(row)

        # append + save incrementally so progress survives interruptions
        pd.DataFrame([row]).to_csv(
            output_path, mode="a", header=not os.path.exists(output_path), index=False
        )